## Perturbazione

In [ ]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath(".."))

from src.datasets.gqa_graph_dataset import GQAGraphDataset, get_graph_dataloader, build_node_vocab, build_rel_vocab
import pandas as pd

In [ ]:
epochs = 20
loss_history = []

In [ ]:
df_val = pd.read_csv("../data/df_val.csv")

all_labels = df_val["labels"].unique()

label2idx = {l: i for i, l in enumerate(all_labels)}
idx2label = {i: l for l, i in label2idx.items()}

df_val["labels"] = df_val["labels"].map(label2idx)

node_vocab = build_node_vocab(df_val)
rel_vocab = build_rel_vocab(df_val)

In [ ]:
import ast
import random
import copy
import pandas as pd


def perturb_graph_objects(
    objects,
    drop_prob=0.2,
    keep_at_least_one=True
):

    objects = copy.deepcopy(objects)

    node_ids = list(objects.keys())

    if len(node_ids) == 0:
        return objects

    # -------------------
    # scegli nodi da tenere
    # -------------------
    kept_nodes = []

    for nid in node_ids:
        if random.random() > drop_prob:
            kept_nodes.append(nid)

    # evita grafo vuoto
    if keep_at_least_one and len(kept_nodes) == 0:
        kept_nodes.append(random.choice(node_ids))

    kept_nodes = set(kept_nodes)

    # -------------------
    # filtra nodi
    # -------------------
    new_objects = {}

    for nid in kept_nodes:

        obj = objects[nid]

        # filtra relazioni
        new_relations = []

        for rel in obj.get("relations", []):

            dst = rel["object"]

            if dst in kept_nodes:
                new_relations.append(rel)

        obj["relations"] = new_relations

        new_objects[nid] = obj

    return new_objects

In [ ]:
def perturb_dataframe(
    df,
    drop_prob=0.2
):

    df = df.copy()

    perturbed_objects = []

    for obj_str in df["objects"]:

        objects = ast.literal_eval(obj_str)

        perturbed = perturb_graph_objects(
            objects,
            drop_prob=drop_prob
        )

        perturbed_objects.append(str(perturbed))

    df["objects"] = perturbed_objects

    return df

In [ ]:
perturbed_df = perturb_dataframe(
    df_val,
    drop_prob=0.3
)

test_dataset_perturbed = GQAGraphDataset(
    perturbed_df,
    node_vocab=node_vocab,
    rel_vocab=rel_vocab,
)

In [ ]:
val_dataset = GQAGraphDataset(
    df=perturbed_df,
    label_col="labels",
    node_vocab=node_vocab,
    rel_vocab=rel_vocab,
    label2idx= label2idx,
    idx2label = idx2label,
    use_bbox=True
)

NUM_CLASSES = len(label2idx)

In [ ]:
from src.models.gnn import GCNEmbeddingNet

model = GCNEmbeddingNet(
    num_node_types=len(node_vocab) + 1,
    emb_dim=128,
    hidden_dim=256,
    num_layers=4,
    dropout=0.2,
    use_bbox=True,
    node_emb_dim=64,
)

In [ ]:
import os
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
gcn_model = GCNEmbeddingNet(
    num_node_types=len(node_vocab) + 1,
    emb_dim=128,
    hidden_dim=256,
    num_layers=4,
    dropout=0.2,
    use_bbox=True,
    node_emb_dim=64,
).to(device)

parent_dir = os.path.abspath("..")
ckpt_path = os.path.join(
    parent_dir,
    "experiments",
    "checkpoints",
    f"gcn{epochs}.pth",
)

from src.training.train_model import load_model_weights

load_model_weights(ckpt_path, gcn_model, device)
gcn_model.eval()

In [ ]:
from src.evaluation.retrieval import (
    gcn_compute_graph_embeddings,
    evaluate_retrieval,
    print_metrics,
    plot_graph_retrieval,
    plot_tsne,
)

In [ ]:
embeddings, labels = gcn_compute_graph_embeddings(
    model=gcn_model,
    dataset=val_dataset,
    device=device,
    batch_size=64,
    num_workers=0,
    normalize=True,
)

print("Embeddings shape:", embeddings.shape)
print("Labels shape:", labels.shape)


results, indices, scores = evaluate_retrieval(
    embeddings,
    labels,
    ks=(1, 5, 10),
)


print_metrics(results)

In [ ]:
IMAGE_DIR = os.path.join(os.path.abspath(".."), "data", "images")

np.random.seed(40)
query_indices = np.random.choice(len(labels), size=5, replace=False)

print(query_indices)
for query_idx in query_indices:


    plot_graph_retrieval(
        df=val_dataset,
        image_dir=IMAGE_DIR,
        labels=labels,
        indices=indices,
        scores=scores,
        query_idx=query_idx,
        idx2label=val_dataset.idx2label,
        topk=5,
        title=f"GCN retrieval (query {query_idx})",
    )

    q_label = labels[query_idx].item()
    print(f"\nQuery {query_idx}: {val_dataset.idx2label.get(q_label, q_label)}")

    for rank, (idx, score) in enumerate(
        zip(indices[query_idx][1:6], scores[query_idx][1:6]),
        start=1
    ):
        lbl = int(labels[idx])
        mark = "OK" if lbl == q_label else "MISS"

        print(
            f"#{rank} idx={idx} score={score:.3f} "
            f"{val_dataset.idx2label.get(lbl, lbl)} [{mark}]"
        )